# LangChain

**One-line mental model of LangChain:** it's a set of standard interfaces (`Runnable`) so that LLMs, prompts, parsers, retrievers, and tools can all be *piped together* with `|` like Unix commands.

Get a free key: https://console.groq.com/keys (takes 30 sec, no card needed)


In [ ]:
!pip install -q langchain langchain-groq langchain-community langchain-chroma langchain-text-splitters sentence-transformers

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")


## Section 1 — The Model (`ChatGroq`)

**Meaning:** LangChain doesn't call the LLM API directly in your code — it wraps every provider (Groq, OpenAI, Anthropic, local) behind the same interface: `.invoke()`. Swap providers, code stays identical. This is the whole point of LangChain existing.


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)

response = llm.invoke("In one sentence, what is Prophet used for in time series forecasting?")
print(type(response))       # AIMessage object, not a raw string
print(response.content)     # the actual text is inside .content


**Try it yourself:** change the prompt to ask something about your Krushna Dairy project. Notice `response` is an `AIMessage` object — it also carries `.response_metadata` (tokens used, etc). Print that too.


In [ ]:
# Your turn — edit and run
print(response.response_metadata)


## Section 2 — PromptTemplate

**Meaning:** Instead of f-strings, LangChain templates are objects — this matters because templates themselves become `Runnable`s you can pipe. `{variable}` is filled in at `.invoke()` time, not at definition time.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a terse AI/ML interview coach."),
    ("human", "Explain {concept} in 2 sentences, aimed at a {level} student."),
])

# A prompt template is NOT text yet — it's a Runnable. .invoke() fills the template.
filled = prompt.invoke({"concept": "XGBoost", "level": "final-year B.Tech"})
print(filled)  # shows the actual list of messages that will be sent


## Section 3 — LCEL: the `|` operator (the part everyone finds confusing)

**Meaning:** Every LangChain component — prompt, model, parser, retriever — implements the same `Runnable` interface (`.invoke()`, `.batch()`, `.stream()`). Because they share that interface, `|` just means "pass my output as your input." A chain built with `|` is called an **LCEL chain** (LangChain Expression Language). There's no magic — it's function composition.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

# prompt -> llm -> parser. Each stage's output becomes the next stage's input.
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"concept": "SHAP values", "level": "recruiter with no ML background"})
print(result)   # now it's a plain string, not an AIMessage — StrOutputParser stripped it


**Try it:** call `chain.batch([...])` with a list of 2-3 different `{"concept":..., "level":...}` dicts — LCEL parallelizes this for you automatically.


In [ ]:
results = chain.batch([
    {"concept": "attention mechanism", "level": "10 year old"},
    {"concept": "WAPE metric", "level": "MSME shop owner"},
])
for r in results:
    print(r, "\n---")


## Section 4 — RunnableParallel & RunnablePassthrough

**Meaning:** Real pipelines aren't always a straight line. `RunnableParallel` runs multiple chains on the *same* input simultaneously and returns a dict of results. `RunnablePassthrough` means "forward the original input untouched" — critical in RAG, where you need both the retrieved docs AND the original question to reach the final prompt.


In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

parallel_chain = RunnableParallel(
    short_answer=prompt | llm | StrOutputParser(),
    original_input=RunnablePassthrough(),
)

out = parallel_chain.invoke({"concept": "Croston's method", "level": "product manager"})
print(out.keys())
print(out["original_input"])


## Section 5 — Structured Output (Pydantic)

**Meaning:** Instead of parsing free text with regex, tell the model to return data matching a schema you define. `with_structured_output` forces the LLM (via tool-calling under the hood) to emit fields that map straight onto a Pydantic model — no manual JSON parsing.


In [ ]:
from pydantic import BaseModel, Field

class SKUForecastNote(BaseModel):
    sku_name: str = Field(description="name of the product SKU")
    risk_level: str = Field(description="low, medium, or high demand volatility")
    reason: str = Field(description="one sentence justification")

structured_llm = llm.with_structured_output(SKUForecastNote)

result = structured_llm.invoke(
    "Ghee has huge demand spikes around Diwali and Sankranti. Classify its forecasting risk."
)
print(result)
print(type(result))       # a real Python object, not a dict of strings
print(result.risk_level)  # dot access


## Section 6 — Memory (multi-turn conversation)

**Meaning:** LLM calls are stateless by default — each `.invoke()` starts fresh. `RunnableWithMessageHistory` wraps a chain so past turns get re-injected automatically, keyed by a `session_id` (so you can run multiple independent conversations).


In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("placeholder", "{history}"),
    ("human", "{input}"),
])
base_chain = chat_prompt | llm | StrOutputParser()

store = {}
def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chat_with_memory = RunnableWithMessageHistory(
    base_chain, get_history,
    input_messages_key="input", history_messages_key="history",
)

cfg = {"configurable": {"session_id": "vedika-session-1"}}
print(chat_with_memory.invoke({"input": "My project is called Krushna Dairy, remember that."}, config=cfg))
print(chat_with_memory.invoke({"input": "What's my project called?"}, config=cfg))  # it remembers


## Section 7 — RAG basics (Retrieval Augmented Generation)

**Meaning:** You already know this conceptually from TA-RAG. In LangChain syntax it's: **load → split → embed → store in a vectorstore → retrieve → stuff into prompt.** We use a tiny in-memory example and fake embeddings here so it runs instantly without downloading a real embedding model — swap in `HuggingFaceEmbeddings` for real projects.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import FakeEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

raw_text = '''Krushna Dairy forecasts 35 SKUs up to 90 days ahead.
It uses Prophet, XGBoost, SARIMAX, and Croston's method depending on SKU sparsity.
A hand-built 46-festival Maharashtra calendar improves accuracy around Diwali and Sankranti.
Portfolio volume-weighted WAPE is around 7 percent.'''

docs = [Document(page_content=raw_text)]

splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=10)
chunks = splitter.split_documents(docs)
print(f"{len(chunks)} chunks created")

# FakeEmbeddings for demo speed — swap for HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2') for real use
embeddings = FakeEmbeddings(size=384)
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

retrieved = retriever.invoke("what models does the forecasting system use")
for r in retrieved:
    print("-", r.page_content)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template(
    "Answer using ONLY this context:\n{context}\n\nQuestion: {question}"
)

def format_docs(docs):
    return "\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is the WAPE of the forecasting system?"))


**Notice the dict syntax** `{"context": ..., "question": ...}` piped into the prompt — that's `RunnableParallel` written in shorthand. This exact pattern (retriever | format | prompt | llm | parser) is the canonical RAG chain you'll see in 90% of LangChain RAG code.


## Section 8 — Tools & basic Agent behavior

**Meaning:** A "tool" is just a Python function the LLM is *told about* (its name, args, docstring) so it can request "please call this function with these arguments" instead of only generating text. The LLM never runs the code — you do, then feed the result back.


In [ ]:
from langchain_core.tools import tool

@tool
def get_sku_risk(sku_name: str) -> str:
    """Return the demand volatility risk level for a given dairy SKU."""
    risk_map = {"ghee": "high", "curd": "low", "paneer": "medium"}
    return risk_map.get(sku_name.lower(), "unknown")

llm_with_tools = llm.bind_tools([get_sku_risk])

response = llm_with_tools.invoke("What's the risk level for ghee?")
print(response.tool_calls)  # the LLM didn't answer directly — it requested a tool call

# You execute the tool call yourself and feed the result back:
if response.tool_calls:
    call = response.tool_calls[0]
    tool_result = get_sku_risk.invoke(call["args"])
    print("Tool result:", tool_result)


This tool-call → execute → feed-back loop, repeated automatically, is *literally* what a LangChain/LangGraph agent is. You've already built multi-agent orchestration in ComplianceOS by hand — this is the primitive underneath it.


## Cheat Sheet — syntax you now know

| Concept | Syntax |
|---|---|
| Call a model | `llm.invoke("text")` → `.content` |
| Prompt template | `ChatPromptTemplate.from_messages([("system",...),("human","{var}")])` |
| Chain (LCEL) | `prompt \| llm \| StrOutputParser()` |
| Parallel branches | `RunnableParallel(a=chain1, b=chain2)` |
| Pass input through | `RunnablePassthrough()` |
| Structured output | `llm.with_structured_output(PydanticModel)` |
| Memory | `RunnableWithMessageHistory(chain, get_history, ...)` |
| RAG retriever | `vectorstore.as_retriever() \| format_docs` |
| Tool | `@tool` decorator + `llm.bind_tools([...])` |

**Next 20-minute rep (do it yourself, no notes):** rebuild Section 7's RAG chain from scratch using your actual TA-RAG or Krushna Dairy project description as the source text. That's the fastest way to make this permanent instead of "seen once."
